In [0]:
# ============================================================
# GOLD CUSTOMER SALES
# Grain: 1 row = 1 customer
# Incremental + MERGE + Watermark
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)
from delta.tables import DeltaTable


# ============================================================
# 1. PATHS
# ============================================================

orders_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "silver/orders/"
)

customers_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "silver/customers/"
)

gold_customer_sales_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "gold/customer_sales/"
)

control_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "metadata/pipeline_watermark/"
)


# ============================================================
# 2. READ SILVER ORDERS
# ============================================================

orders_df = (
    spark.read
    .format("delta")
    .load(orders_path)
)

print("Silver Orders:", orders_df.count())


# ============================================================
# 3. READ SILVER CUSTOMERS
# ============================================================

customers_df = (
    spark.read
    .format("delta")
    .load(customers_path)
)

print("Silver Customers:", customers_df.count())


# ============================================================
# 4. VALIDATE SCD2 CURRENT CUSTOMER UNIQUENESS
# ============================================================

current_customers_df = (
    customers_df
    .filter(
        F.col("is_current") == True
    )
)

duplicate_current_customers_df = (
    current_customers_df
    .groupBy("customer_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_current_count = (
    duplicate_current_customers_df.count()
)

print(
    "Duplicate current customer IDs:",
    duplicate_current_count
)

if duplicate_current_count > 0:

    print(
        "ERROR: Silver Customer SCD2 contains multiple "
        "is_current=true rows for the same customer_id."
    )

    display(
        duplicate_current_customers_df
        .orderBy("customer_id")
    )

    raise Exception(
        "SCD2 validation failed: "
        "multiple current records exist for the same customer_id."
    )


# ============================================================
# 5. READ CONTROL TABLE
# ============================================================

control_df = (
    spark.read
    .format("delta")
    .load(control_path)
)


# ============================================================
# 6. CREATE GOLD CUSTOMER WATERMARK IF NEEDED
# ============================================================

gold_customer_exists = (
    control_df
    .filter(
        F.col("pipeline_name") == "gold_customer_sales"
    )
    .count() > 0
)

print(
    "gold_customer_sales watermark exists:",
    gold_customer_exists
)


gold_customer_control_schema = StructType([
    StructField(
        "pipeline_name",
        StringType(),
        False
    ),
    StructField(
        "last_processed_timestamp",
        TimestampType(),
        True
    ),
    StructField(
        "last_run_id",
        StringType(),
        True
    ),
    StructField(
        "updated_at",
        TimestampType(),
        True
    )
])


if not gold_customer_exists:

    gold_customer_control_df = spark.createDataFrame(
        [
            (
                "gold_customer_sales",
                None,
                None,
                None
            )
        ],
        schema=gold_customer_control_schema
    )

    (
        gold_customer_control_df
        .write
        .format("delta")
        .mode("append")
        .save(control_path)
    )

    print(
        "Created gold_customer_sales watermark entry."
    )


# ============================================================
# 7. READ GOLD CUSTOMER WATERMARK
# ============================================================

gold_customer_control_df = (
    spark.read
    .format("delta")
    .load(control_path)
    .filter(
        F.col("pipeline_name") == "gold_customer_sales"
    )
)

gold_customer_last_processed_timestamp = (
    gold_customer_control_df
    .select("last_processed_timestamp")
    .collect()[0][0]
)

print(
    "Gold Customer last processed timestamp:",
    gold_customer_last_processed_timestamp
)


# ============================================================
# 8. IDENTIFY INCREMENTAL ORDERS
# ============================================================

if gold_customer_last_processed_timestamp is None:

    incremental_orders_df = orders_df

else:

    incremental_orders_df = (
        orders_df
        .filter(
            F.col("ingestion_timestamp")
            > gold_customer_last_processed_timestamp
        )
    )


incremental_count = incremental_orders_df.count()

print(
    "Incremental orders:",
    incremental_count
)


# ============================================================
# 9. NO NEW DATA → SUCCESSFUL NO-OP
# ============================================================

if incremental_count == 0:

    print(
        "No new orders found."
    )

    print(
        "Gold Customer Sales is already up to date."
    )

    print(
        "Watermark will NOT be updated."
    )

    dbutils.notebook.exit(
        "SUCCESS: No new records."
    )


# ============================================================
# 10. FIND AFFECTED CUSTOMERS
# ============================================================

affected_customers_df = (
    incremental_orders_df
    .select("customer_id")
    .distinct()
)

print(
    "Affected customers:",
    affected_customers_df.count()
)


# ============================================================
# 11. GET ALL SILVER ORDERS FOR AFFECTED CUSTOMERS
# ============================================================

affected_customer_orders_df = (
    orders_df.alias("o")
    .join(
        affected_customers_df.alias("c"),
        F.col("o.customer_id")
        == F.col("c.customer_id"),
        "inner"
    )
    .select(
        "o.*"
    )
)

print(
    "Orders belonging to affected customers:",
    affected_customer_orders_df.count()
)


# ============================================================
# 12. CALCULATE COMPLETE CUSTOMER METRICS
# Grain = 1 row per customer
# ============================================================

customer_metrics_input_df = (
    affected_customer_orders_df
    .select(
        "customer_id",
        "order_id",
        "sales_amount",
        "quantity"
    )
)

affected_customer_metrics_df = (
    customer_metrics_input_df
    .groupBy("customer_id")
    .agg(
        F.sum("sales_amount").alias(
            "total_sales"
        ),

        F.countDistinct("order_id").alias(
            "total_orders"
        ),

        F.sum("quantity").alias(
            "total_quantity"
        ),

        F.round(
            F.avg("sales_amount"),
            2
        ).alias(
            "average_order_value"
        )
    )
)

print(
    "Customer metric rows:",
    affected_customer_metrics_df.count()
)


# ============================================================
# 13. JOIN CURRENT CUSTOMER ATTRIBUTES
# ============================================================

affected_customer_gold_df = (
    affected_customer_metrics_df.alias("g")
    .join(
        current_customers_df.alias("c"),
        F.col("g.customer_id")
        == F.col("c.customer_id"),
        "left"
    )
    .select(
        F.col("g.customer_id").alias(
            "customer_id"
        ),

        F.col("c.name").alias(
            "customer_name"
        ),

        F.col("c.city").alias(
            "customer_city"
        ),

        F.col("c.market").alias(
            "customer_market"
        ),

        F.col("g.total_sales").alias(
            "total_sales"
        ),

        F.col("g.total_orders").alias(
            "total_orders"
        ),

        F.col("g.total_quantity").alias(
            "total_quantity"
        ),

        F.col("g.average_order_value").alias(
            "average_order_value"
        )
    )
)


# ============================================================
# 14. FINAL SOURCE GRAIN VALIDATION
# ============================================================

duplicate_gold_source_df = (
    affected_customer_gold_df
    .groupBy("customer_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_gold_source_count = (
    duplicate_gold_source_df.count()
)

print(
    "Duplicate customer IDs in Gold source:",
    duplicate_gold_source_count
)

if duplicate_gold_source_count > 0:

    display(
        duplicate_gold_source_df
        .orderBy("customer_id")
    )

    raise Exception(
        "Gold source grain violation: "
        "customer_id must be unique before MERGE."
    )


# ============================================================
# 15. DISPLAY GOLD SOURCE
# ============================================================

display(
    affected_customer_gold_df
    .orderBy("customer_id")
)


# ============================================================
# 16. CREATE OR MERGE GOLD TABLE
# ============================================================

gold_table_exists = DeltaTable.isDeltaTable(
    spark,
    gold_customer_sales_path
)

print(
    "Gold Customer Sales table exists:",
    gold_table_exists
)


if not gold_table_exists:

    print(
        "Creating initial Gold Customer Sales table."
    )

    (
        affected_customer_gold_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(gold_customer_sales_path)
    )

else:

    print(
        "Gold Customer Sales table exists. "
        "Running MERGE."
    )

    gold_table = DeltaTable.forPath(
        spark,
        gold_customer_sales_path
    )

    (
        gold_table.alias("target")
        .merge(
            affected_customer_gold_df.alias("source"),
            "target.customer_id = source.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


# ============================================================
# 17. GET NEW WATERMARK
# ============================================================

gold_customer_processed_timestamp = (
    incremental_orders_df
    .agg(
        F.max("ingestion_timestamp")
        .alias("max_ingestion_timestamp")
    )
    .collect()[0]["max_ingestion_timestamp"]
)

print(
    "New Gold Customer watermark:",
    gold_customer_processed_timestamp
)


# ============================================================
# 18. PREPARE WATERMARK
# ============================================================

gold_customer_watermark_schema = StructType([
    StructField(
        "pipeline_name",
        StringType(),
        False
    ),
    StructField(
        "last_processed_timestamp",
        TimestampType(),
        True
    ),
    StructField(
        "last_run_id",
        StringType(),
        True
    ),
    StructField(
        "updated_at",
        TimestampType(),
        True
    )
])


gold_customer_watermark_df = (
    spark.createDataFrame(
        [
            (
                "gold_customer_sales",
                gold_customer_processed_timestamp,
                "GOLD_CUSTOMER_SALES_RUN",
                None
            )
        ],
        schema=gold_customer_watermark_schema
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)


# ============================================================
# 19. UPDATE WATERMARK
# ============================================================

control_table = DeltaTable.forPath(
    spark,
    control_path
)

(
    control_table.alias("target")
    .merge(
        gold_customer_watermark_df.alias("source"),
        "target.pipeline_name = source.pipeline_name"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)


# ============================================================
# 20. FINAL VERIFICATION
# ============================================================

print(
    "Final Gold Customer Sales:"
)

display(
    spark.read
    .format("delta")
    .load(gold_customer_sales_path)
    .orderBy("customer_id")
)

print(
    "Final Gold Customer watermark:"
)

display(
    spark.read
    .format("delta")
    .load(control_path)
    .filter(
        F.col("pipeline_name")
        == "gold_customer_sales"
    )
)

In [0]:
display(
    spark.read
    .format("delta")
    .load(gold_customer_sales_path)
    
)

customer_id,customer_name,customer_city,customer_market,total_sales,total_orders,total_quantity,average_order_value
180574,Customer Oct7 20574,Canberra,Australia,2057.91,1,5,2057.91
167595,Customer Oct7 7595,Hamburg,Germany,1272.99,1,3,1272.99
184969,Customer Oct7 24969,Canberra,Australia,2318.46,1,5,2318.46
165021,Customer Oct7 5021,Lucknow,India,397.05,1,1,397.05
175649,Customer Oct7 15649,Canberra,Australia,1374.7,1,5,1374.7
167647,Customer Oct7 7647,Ottawa,Canada,1505.71,1,4,1505.71
181971,Customer Oct7 21971,Lucknow,India,55.5,1,1,55.5
168521,Customer Oct7 8521,Lucknow,India,365.55,1,1,365.55
170591,Customer Oct7 10591,Lucknow,India,431.52,1,1,431.52
169619,Customer Oct7 9619,Canberra,Australia,2240.91,1,5,2240.91
